Step 1 — Extract Relevant Content Features

In [12]:
import pandas as pd

df = pd.read_csv("clean_netflix.csv")

print(df.columns.tolist())

['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']


In [13]:
features = df[["title", "type", "listed_in"]]

print(features.head())

                              title     type  \
0              Dick Johnson Is Dead    Movie   
1                         Ganglands  TV Show   
2                     Midnight Mass  TV Show   
3  Confessions of an Invisible Girl    Movie   
4                           Sankofa    Movie   

                                           listed_in  
0                                      Documentaries  
1  Crime TV Shows, International TV Shows, TV Act...  
2                 TV Dramas, TV Horror, TV Mysteries  
3                 Children & Family Movies, Comedies  
4   Dramas, Independent Movies, International Movies  


Step 2 — Text Preprocessing

In [14]:
features["content"] = (
    features["type"] + " " + features["listed_in"]
)

print(features[["title", "content"]].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                             content  
0                                Movie Documentaries  
1  TV Show Crime TV Shows, International TV Shows...  
2         TV Show TV Dramas, TV Horror, TV Mysteries  
3           Movie Children & Family Movies, Comedies  
4  Movie Dramas, Independent Movies, Internationa...  


In [15]:
features["content"] = features["content"].str.lower()

print(features[["title", "content"]].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                             content  
0                                movie documentaries  
1  tv show crime tv shows, international tv shows...  
2         tv show tv dramas, tv horror, tv mysteries  
3           movie children & family movies, comedies  
4  movie dramas, independent movies, internationa...  


In [16]:
features["content"] = (
    features["content"]
    .str.replace(",", " ", regex=False)
    .str.lower()
)

print(features[["title", "content"]].head())

                              title  \
0              Dick Johnson Is Dead   
1                         Ganglands   
2                     Midnight Mass   
3  Confessions of an Invisible Girl   
4                           Sankofa   

                                             content  
0                                movie documentaries  
1  tv show crime tv shows  international tv shows...  
2         tv show tv dramas  tv horror  tv mysteries  
3           movie children & family movies  comedies  
4  movie dramas  independent movies  internationa...  


Step 3 — Calculate Content Similarity

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(features["content"])

print(tfidf_matrix.shape)

(8790, 47)


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

print(similarity.shape)

(8790, 8790)


Step 4 — Generate Recommendations 

In [19]:
def recommend(title, n=5):
    idx = features.index[features["title"].str.lower() == title.lower()][0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:n+1]

    recommended_indices = [i[0] for i in scores]

    return features.iloc[recommended_indices][["title", "type", "listed_in"]]

In [20]:
recommend("Sankofa")

,title,type,listed_in
54,Ankahi Kahaniya,Movie,"Dramas, Independent Movies, International Movies"
280,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies"
382,La diosa del asfalto,Movie,"Dramas, Independent Movies, International Movies"
509,Eyes of a Thief,Movie,"Dramas, Independent Movies, International Movies"
510,Pomegranates and Myrrh,Movie,"Dramas, Independent Movies, International Movies"


In [21]:
def recommend(title, n=5):
    idx = features.index[features["title"].str.lower() == title.lower()][0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:n+1]

    recommendations = []

    for i, score in scores:
        recommendations.append([
            features.iloc[i]["title"],
            features.iloc[i]["type"],
            features.iloc[i]["listed_in"],
            round(score, 2)
        ])

    return pd.DataFrame(
        recommendations,
        columns=["title", "type", "listed_in", "similarity_score"]
    )

In [22]:
recommend("Sankofa")

,title,type,listed_in,similarity_score
0,Ankahi Kahaniya,Movie,"Dramas, Independent Movies, International Movies",1.0
1,"LSD: Love, Sex Aur Dhokha",Movie,"Dramas, Independent Movies, International Movies",1.0
2,La diosa del asfalto,Movie,"Dramas, Independent Movies, International Movies",1.0
3,Eyes of a Thief,Movie,"Dramas, Independent Movies, International Movies",1.0
4,Pomegranates and Myrrh,Movie,"Dramas, Independent Movies, International Movies",1.0


Step 5 — Evaluate Recommendation Quality

In [23]:
selected_title = "Sankofa"

recommended = recommend(selected_title)

selected_categories = set(
    features.loc[
        features["title"].str.lower() == selected_title.lower(),
        "listed_in"
    ].iloc[0].split(", ")
)

recommended["category_overlap"] = recommended["listed_in"].apply(
    lambda x: len(selected_categories.intersection(set(x.split(", "))))
)

print(recommended)

                       title   type  \
0            Ankahi Kahaniya  Movie   
1  LSD: Love, Sex Aur Dhokha  Movie   
2       La diosa del asfalto  Movie   
3            Eyes of a Thief  Movie   
4     Pomegranates and Myrrh  Movie   

                                          listed_in  similarity_score  \
0  Dramas, Independent Movies, International Movies               1.0   
1  Dramas, Independent Movies, International Movies               1.0   
2  Dramas, Independent Movies, International Movies               1.0   
3  Dramas, Independent Movies, International Movies               1.0   
4  Dramas, Independent Movies, International Movies               1.0   

   category_overlap  
0                 3  
1                 3  
2                 3  
3                 3  
4                 3  


# Task 3 — Recommendation System Analysis

## Objective

The objective of this task was to develop a basic content-based recommendation system using Netflix titles and categories.

## Work Completed

### 1. Feature Extraction

Selected relevant features from the cleaned Netflix dataset:

* `title`
* `type`
* `listed_in`

### 2. Text Preprocessing

Combined `type` and `listed_in` into a single content feature. The text was cleaned by removing commas and converting it to lowercase.

### 3. TF-IDF Feature Extraction

Used **TF-IDF Vectorization** to convert the text-based content into numerical vectors.

Resulting matrix:

**8,790 titles × 47 features**

### 4. Content Similarity

Used **Cosine Similarity** to compare the content vectors of all Netflix titles.

Similarity matrix:

**8,790 × 8,790**

### 5. Recommendation Generation

Created a recommendation function that takes a selected Netflix title and returns the top 5 most similar titles.

For example, for **Sankofa**, the system recommended:

* Ankahi Kahaniya
* LSD: Love, Sex Aur Dhokha
* La diosa del asfalto
* Eyes of a Thief
* Pomegranates and Myrrh

### 6. Evaluation

Evaluated the recommendations using similarity scores and category overlap.

For the selected title **Sankofa**, all five recommendations had:

* Similarity Score: **1.0**
* Category Overlap: **3**

## Conclusion

A basic content-based Netflix recommendation system was successfully implemented using **TF-IDF and Cosine Similarity**. The system can identify and recommend titles with similar content categories.

**Tools Used:** Python, Pandas, Scikit-learn
